<a href="https://colab.research.google.com/github/travistan101/linkedin-salary-predictor/blob/main/01_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LinkedIn Job Postings: Salary Prediction & Feature Engineering
---
## 1. Environment & Setup
In this section, we mount Google Drive to access our datasets and install the necessary dependencies, including PySpark for big data processing and OpenAI/LangGraph for our agentic feature selection later on.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install pyspark openai python-dotenv langgraph langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 25.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


## 2. Comprehensive Data Cleaning & Exploratory Data Analysis (EDA)
Here, we load our primary tables (`postings`, `benefits`, `companies`, `employee_counts`) and perform a deep dive into the data.

**Key Goals of this section:**
* **Standardize** column types and cast strings to numeric/timestamp types safely.
* **Audit** the target variable (salary), identifying missingness and overlapping columns (`min_salary`, `max_salary`, `med_salary`, `normalized_salary`).
* **Annualize** the salary data so hourly, monthly, and yearly roles are comparable.
* **Join** the supporting tables to the main postings dataset safely to avoid row duplication.

In [ ]:
# ============================================
# STRONG SALARY-FOCUSED EDA FOR LINKEDIN JOBS
# ============================================
# prioritises the tables:
#   1) postings.csv
#   2) benefits.csv
#   3) companies.csv
#   4) employee_counts.csv
#
# Main goals:
# - Clean and standardise key columns
# - Audit salary target quality
# - Compare salary target candidates
# - Annualize salary for comparability
# - Inspect pay_period and currency
# - Check missingness / outliers
# - Run salary-vs-feature EDA
# - Safely join supporting tables
# - Avoid accidental row duplication
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("LinkedInSalaryEDA").getOrCreate()

# -----------------------------
# 1. PATHS
# -----------------------------

POSTINGS_PATH = "/content/drive/MyDrive/BT4221 Group 13/Dataset/postings.csv"
BENEFITS_PATH = "/content/drive/MyDrive/BT4221 Group 13/Dataset/jobs/benefits.csv"
COMPANIES_PATH = "/content/drive/MyDrive/BT4221 Group 13/Dataset/companies/companies.csv"
EMPLOYEE_COUNTS_PATH = "/content/drive/MyDrive/BT4221 Group 13/Dataset/companies/employee_counts.csv"

# -----------------------------
# 2. GENERIC CSV READER
# -----------------------------
def read_csv(path):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("quote", '"')
        .csv(path)
    )

# -----------------------------
# 3. HELPER CASTING FUNCTIONS
# -----------------------------


def cast_numeric(df, col_name, dtype="double"):
    string_col = F.trim(F.col(col_name).cast("string"))
    is_integer = dtype in {"long", "int", "integer", "bigint"}
    pattern_strip = r"[^0-9\-]" if is_integer else r"[^0-9.\-]"
    cleaned = F.regexp_replace(string_col, pattern_strip, "")

    if is_integer:
        valid_int = cleaned.rlike(r"^-?\d+$")
        abs_digits = F.regexp_replace(cleaned, r"^-", "")
        safe_int = valid_int & (F.length(abs_digits) <= F.lit(18))
        return df.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), F.lit(None))
             .when(F.length(cleaned) == 0, F.lit(None))
             .when(safe_int, cleaned)
             .otherwise(F.lit(None))
             .cast(dtype)
        )

    valid_double = cleaned.rlike(r"^-?(\d+(\.\d+)?|\.\d+)$")
    return df.withColumn(
        col_name,
        F.when(F.col(col_name).isNull(), F.lit(None))
         .when(F.length(cleaned) == 0, F.lit(None))
         .when(valid_double, cleaned)
         .otherwise(F.lit(None))
         .cast(dtype)
    )

def cast_boolean_strict(df, col_name):

    # It only accepts explicit boolean-like tokens and strict 0/1 style values.
    # avoids turning numeric into True.
    raw = F.lower(F.trim(F.col(col_name).cast("string")))
    return df.withColumn(
        col_name,
        F.when(raw.isin("true", "t", "yes", "y", "1", "1.0", "1.00"), F.lit(True))
         .when(raw.isin("false", "f", "no", "n", "0", "0.0", "0.00"), F.lit(False))
         .otherwise(F.lit(None).cast("boolean"))
    )

def cast_timestamp_multi(df, col_name):
    raw = F.trim(F.col(col_name).cast("string"))
    num_clean = F.regexp_replace(raw, r"[^0-9.\-]", "")
    num_val = F.when(
        num_clean.rlike(r"^-?(\d+(\.\d+)?|\.\d+)$"),
        num_clean.cast("double")
    ).otherwise(F.lit(None).cast("double"))

    epoch_ts = (
        F.when(num_val.isNull(), F.lit(None).cast("timestamp"))
         .when(num_val >= F.lit(1e12), (num_val / F.lit(1000)).cast("timestamp"))
         .when((num_val >= F.lit(1e9)) & (num_val < F.lit(1e12)), num_val.cast("timestamp"))
         .otherwise(F.lit(None).cast("timestamp"))
    )

    return df.withColumn(
        col_name,
        F.coalesce(
            F.try_to_timestamp(raw, F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp(raw, F.lit("yyyy-MM-dd'T'HH:mm:ss")),
            F.try_to_timestamp(raw, F.lit("yyyy-MM-dd")),
            epoch_ts
        )
    )

# -----------------------------
# 4. LOAD TABLES
# -----------------------------
postings_raw = read_csv(POSTINGS_PATH)
benefits_raw = read_csv(BENEFITS_PATH)
companies_raw = read_csv(COMPANIES_PATH)
employee_counts_raw = read_csv(EMPLOYEE_COUNTS_PATH)

print("=== Raw Table Shapes ===")
print("postings rows:", postings_raw.count(), "cols:", len(postings_raw.columns))
print("benefits rows:", benefits_raw.count(), "cols:", len(benefits_raw.columns))
print("companies rows:", companies_raw.count(), "cols:", len(companies_raw.columns))
print("employee_counts rows:", employee_counts_raw.count(), "cols:", len(employee_counts_raw.columns))

# -----------------------------
# 5. CLEAN POSTINGS TABLE
# -----------------------------
# clean the key columns needed for salary EDA.
postings_df = postings_raw

for c in ["job_id", "company_id", "views", "applies", "zip_code", "fips"]:
    if c in postings_df.columns:
        postings_df = cast_numeric(postings_df, c, "long")

for c in ["max_salary", "med_salary", "min_salary", "normalized_salary"]:
    if c in postings_df.columns:
        postings_df = cast_numeric(postings_df, c, "double")

for c in ["remote_allowed", "sponsored"]:
    if c in postings_df.columns:
        postings_df = cast_boolean_strict(postings_df, c)

for c in ["original_listed_time", "listed_time", "expiry", "closed_time"]:
    if c in postings_df.columns:
        postings_df = cast_timestamp_multi(postings_df, c)

for c in [
    "title", "description", "skills_desc", "location", "pay_period", "currency",
    "formatted_work_type", "formatted_experience_level", "work_type",
    "application_type", "compensation_type", "posting_domain"
]:
    if c in postings_df.columns:
        postings_df = postings_df.withColumn(c, F.trim(F.col(c)))

if "pay_period" in postings_df.columns:
    postings_df = postings_df.withColumn("pay_period", F.upper(F.col("pay_period")))

if "currency" in postings_df.columns:
    postings_df = postings_df.withColumn("currency", F.upper(F.col("currency")))

# Remove exact duplicate job_ids if present.
if "job_id" in postings_df.columns:
    postings_df = postings_df.dropDuplicates(["job_id"])

print("\n=== Cleaned postings schema ===")
postings_df.printSchema()

# -----------------------------
# 6. CLEAN SUPPORTING TABLES
# -----------------------------
benefits_df = benefits_raw
if "job_id" in benefits_df.columns:
    benefits_df = cast_numeric(benefits_df, "job_id", "long")
if "type" in benefits_df.columns:
    benefits_df = benefits_df.withColumn("type", F.trim(F.col("type")))
if "inferred" in benefits_df.columns:
    benefits_df = cast_boolean_strict(benefits_df, "inferred")

companies_df = companies_raw
if "company_id" in companies_df.columns:
    companies_df = cast_numeric(companies_df, "company_id", "long")
if "company_size" in companies_df.columns:
    companies_df = cast_numeric(companies_df, "company_size", "long")
for c in ["name", "description", "country", "state", "city", "zip_code", "address", "url"]:
    if c in companies_df.columns:
        companies_df = companies_df.withColumn(c, F.trim(F.col(c)))

employee_counts_df = employee_counts_raw
for c in ["company_id", "employee_count", "follower_count", "time_recorded"]:
    if c in employee_counts_df.columns:
        employee_counts_df = cast_numeric(employee_counts_df, c, "long")

# -----------------------------
# 7. QUICK NULL PROFILE FOR POSTINGS
# -----------------------------
print("\n=== Top 25 null percentages in postings ===")
postings_count = postings_df.count()

null_exprs = [
    (F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.lit(max(postings_count, 1)) * 100).alias(c)
    for c in postings_df.columns
]
null_row = postings_df.agg(*null_exprs).collect()[0].asDict()
null_rows = [(c, float(null_row.get(c) or 0.0)) for c in postings_df.columns]

spark.createDataFrame(null_rows, ["column", "null_pct"]) \
    .orderBy(F.desc("null_pct")) \
    .show(25, truncate=False)

# -----------------------------
# 8. SALARY TARGET AUDIT
# -----------------------------
print("\n=== Salary Target Audit ===")

salary_cols_present = [c for c in ["min_salary", "med_salary", "max_salary", "normalized_salary"] if c in postings_df.columns]
postings_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNotNull().cast("int")).alias(f"{c}_nonnull")
        for c in salary_cols_present
    ]
).show(truncate=False)

if all(c in postings_df.columns for c in ["min_salary", "max_salary"]):
    postings_df.select(
        F.sum((F.col("min_salary") > F.col("max_salary")).cast("int")).alias("rows_min_gt_max")
    ).show(truncate=False)

if all(c in postings_df.columns for c in ["med_salary", "min_salary"]):
    postings_df.select(
        F.sum(
            (F.col("med_salary").isNotNull() &
             F.col("min_salary").isNotNull() &
             (F.col("med_salary") < F.col("min_salary"))).cast("int")
        ).alias("rows_med_lt_min")
    ).show(truncate=False)

if all(c in postings_df.columns for c in ["med_salary", "max_salary"]):
    postings_df.select(
        F.sum(
            (F.col("med_salary").isNotNull() &
             F.col("max_salary").isNotNull() &
             (F.col("med_salary") > F.col("max_salary"))).cast("int")
        ).alias("rows_med_gt_max")
    ).show(truncate=False)

# -----------------------------
# 9. CONSTRUCT TARGET CANDIDATES
# -----------------------------
# We compare:
# 1) normalized_salary
# 2) med_salary
# 3) midpoint of min/max
# 4) fallback target candidate
postings_df = postings_df.withColumn(
    "range_mid_salary",
    F.when(
        F.col("min_salary").isNotNull() & F.col("max_salary").isNotNull(),
        (F.col("min_salary") + F.col("max_salary")) / 2.0
    )
)

postings_df = postings_df.withColumn(
    "salary_target_raw",
    F.when(F.col("med_salary").isNotNull(), F.col("med_salary"))
     .when(F.col("range_mid_salary").isNotNull(), F.col("range_mid_salary"))
)

candidate_exprs = []
for c in ["normalized_salary", "med_salary", "range_mid_salary", "salary_target_raw"]:
    if c in postings_df.columns:
        candidate_exprs.append(F.sum(F.col(c).isNotNull().cast("int")).alias(f"{c}_nonnull"))

print("\n=== Target Candidate Coverage ===")
postings_df.select(F.count("*").alias("rows"), *candidate_exprs).show(truncate=False)

# -----------------------------
# 10. PAY PERIOD + CURRENCY AUDIT
# -----------------------------
print("\n=== Pay Period Distribution ===")
if "pay_period" in postings_df.columns:
    postings_df.groupBy("pay_period").count().orderBy(F.desc("count")).show(20, truncate=False)

print("\n=== Currency Distribution ===")
if "currency" in postings_df.columns:
    postings_df.groupBy("currency").count().orderBy(F.desc("count")).show(20, truncate=False)

print("\n=== Currency Among Salary Rows ===")
if "currency" in postings_df.columns:
    postings_df.where(
        F.col("salary_target_raw").isNotNull() | F.col("normalized_salary").isNotNull()
    ).groupBy("currency").count().orderBy(F.desc("count")).show(20, truncate=False)

# -----------------------------
# 11. ANNUALIZE SALARY CANDIDATES
# -----------------------------
# This is non-negotiable if pay_period mixes hourly/monthly/yearly.
def annualize_from_period(salary_col):
    return (
        F.when(F.col("pay_period") == "YEARLY", F.col(salary_col))
         .when(F.col("pay_period") == "MONTHLY", F.col(salary_col) * 12.0)
         .when(F.col("pay_period") == "HOURLY", F.col(salary_col) * 2080.0)
    )

if "salary_target_raw" in postings_df.columns:
    postings_df = postings_df.withColumn("annual_salary_from_raw", annualize_from_period("salary_target_raw"))

if "normalized_salary" in postings_df.columns:
    # audit it separately
    postings_df = postings_df.withColumn("normalized_salary_candidate", F.col("normalized_salary"))

# Choose an EDA target candidate column for inspection.
# compare both if both exist.
print("\n=== Annualized Target Coverage ===")
exprs = [F.count("*").alias("rows")]
if "annual_salary_from_raw" in postings_df.columns:
    exprs.append(F.sum(F.col("annual_salary_from_raw").isNotNull().cast("int")).alias("annual_salary_from_raw_nonnull"))
if "normalized_salary_candidate" in postings_df.columns:
    exprs.append(F.sum(F.col("normalized_salary_candidate").isNotNull().cast("int")).alias("normalized_salary_candidate_nonnull"))

postings_df.select(*exprs).show(truncate=False)

# -----------------------------
# 12. COMPARE NORMALIZED_SALARY VS RECONSTRUCTED ANNUAL SALARY
# -----------------------------
print("\n=== Compare normalized_salary vs annual_salary_from_raw ===")
if all(c in postings_df.columns for c in ["normalized_salary_candidate", "annual_salary_from_raw"]):
    compare_df = postings_df.where(
        F.col("normalized_salary_candidate").isNotNull() &
        F.col("annual_salary_from_raw").isNotNull()
    ).withColumn(
        "salary_abs_diff",
        F.abs(F.col("normalized_salary_candidate") - F.col("annual_salary_from_raw"))
    ).withColumn(
        "salary_pct_diff",
        F.when(
            F.col("annual_salary_from_raw") != 0,
            F.abs(F.col("normalized_salary_candidate") - F.col("annual_salary_from_raw")) / F.abs(F.col("annual_salary_from_raw"))
        )
    )

    compare_df.select(
        F.count("*").alias("rows_with_both"),
        F.avg("salary_abs_diff").alias("avg_abs_diff"),
        F.expr("percentile_approx(salary_abs_diff, 0.5)").alias("median_abs_diff"),
        F.avg("salary_pct_diff").alias("avg_pct_diff"),
        F.expr("percentile_approx(salary_pct_diff, 0.5)").alias("median_pct_diff")
    ).show(truncate=False)

    compare_df.select(
        "job_id", "title", "pay_period", "min_salary", "med_salary", "max_salary",
        "annual_salary_from_raw", "normalized_salary_candidate", "salary_abs_diff", "salary_pct_diff"
    ).orderBy(F.desc("salary_abs_diff")).show(30, truncate=False)

# -----------------------------
# 13. PICK AN EDA SALARY COLUMN FOR ANALYSIS
# -----------------------------
# Strategy:
# - if normalized_salary exists, inspect it carefully
# - otherwise use annual_salary_from_raw
# - if both exist, prefer normalized_salary ONLY if it looks sensible after audit
#
# Here we create both and keep them visible.
# For grouped EDA below, we use annual_salary_eda with a fallback.
postings_df = postings_df.withColumn(
    "annual_salary_eda",
    F.when(F.col("normalized_salary_candidate").isNotNull(), F.col("normalized_salary_candidate"))
     .otherwise(F.col("annual_salary_from_raw"))
)

postings_df = postings_df.withColumn(
    "log_annual_salary_eda",
    F.when(F.col("annual_salary_eda") > 0, F.log1p(F.col("annual_salary_eda")))
)

print("\n=== EDA Salary Target Summary ===")
postings_df.select(
    F.count("*").alias("rows"),
    F.sum(F.col("annual_salary_eda").isNotNull().cast("int")).alias("annual_salary_eda_nonnull"),
    F.sum((F.col("annual_salary_eda") <= 0).cast("int")).alias("annual_salary_eda_nonpositive")
).show(truncate=False)

# -----------------------------
# 14. SALARY DISTRIBUTION + QUANTILES
# -----------------------------
print("\n=== Annual Salary Distribution ===")
postings_df.where(F.col("annual_salary_eda").isNotNull()).select(
    F.min("annual_salary_eda").alias("min_salary"),
    F.max("annual_salary_eda").alias("max_salary"),
    F.avg("annual_salary_eda").alias("avg_salary"),
    F.expr("percentile_approx(annual_salary_eda, array(0.01,0.05,0.25,0.5,0.75,0.95,0.99))").alias("salary_quantiles"),
    F.expr("percentile_approx(log_annual_salary_eda, array(0.01,0.05,0.25,0.5,0.75,0.95,0.99))").alias("log_salary_quantiles")
).show(truncate=False)

# -----------------------------
# 15. OUTLIER AUDIT
# -----------------------------
print("\n=== Global Salary Outlier Audit ===")
salary_bounds = postings_df.where(F.col("annual_salary_eda").isNotNull()).select(
    F.expr("percentile_approx(annual_salary_eda, 0.25)").alias("q1"),
    F.expr("percentile_approx(annual_salary_eda, 0.75)").alias("q3")
).collect()[0]

q1 = salary_bounds["q1"]
q3 = salary_bounds["q3"]
iqr = q3 - q1 if q1 is not None and q3 is not None else None

if iqr is not None:
    lower_bound = max(0, q1 - 1.5 * iqr)
    upper_bound = q3 + 1.5 * iqr

    print(f"Q1 = {q1}")
    print(f"Q3 = {q3}")
    print(f"IQR = {iqr}")
    print(f"Lower bound = {lower_bound}")
    print(f"Upper bound = {upper_bound}")

    outlier_df = postings_df.where(
        F.col("annual_salary_eda").isNotNull() &
        ((F.col("annual_salary_eda") < lower_bound) | (F.col("annual_salary_eda") > upper_bound))
    )

    print("Outlier count:")
    print(outlier_df.count())

    outlier_cols = [c for c in [
        "job_id", "title", "location", "currency", "pay_period",
        "annual_salary_eda", "normalized_salary_candidate", "annual_salary_from_raw"
    ] if c in outlier_df.columns]
    outlier_df.select(*outlier_cols).show(30, truncate=False)

# -----------------------------
# 16. MISSINGNESS IN SALARY-USABLE SUBSET
# -----------------------------

print("\n=== Missingness in Salary-Usable Subset ===")
salary_subset = postings_df.where(F.col("annual_salary_eda").isNotNull() & (F.col("annual_salary_eda") > 0))
salary_subset_count = salary_subset.count()

subset_null_exprs = [
    (F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.lit(max(salary_subset_count, 1)) * 100).alias(c)
    for c in salary_subset.columns
]
subset_null_row = salary_subset.agg(*subset_null_exprs).collect()[0].asDict()
subset_null_rows = [(c, float(subset_null_row.get(c) or 0.0)) for c in salary_subset.columns]

spark.createDataFrame(subset_null_rows, ["column", "null_pct_salary_subset"]) \
    .orderBy(F.desc("null_pct_salary_subset")) \
    .show(30, truncate=False)

# -----------------------------
# 17. TEXT COVERAGE / TEXT RICHNESS
# -----------------------------
# This checks whether text columns are actually populated enough to be useful.
print("\n=== Text Coverage / Richness ===")
for c in ["title", "description", "skills_desc"]:
    if c in postings_df.columns:
        postings_df = postings_df.withColumn(f"{c}_len", F.length(F.coalesce(F.col(c), F.lit(""))))
        postings_df = postings_df.withColumn(
            f"{c}_word_count",
            F.size(F.split(F.trim(F.coalesce(F.col(c), F.lit(""))), r"\s+"))
        )

        print(f"\n--- Text summary for {c} ---")
        postings_df.select(
            F.count("*").alias("rows"),
            F.sum((F.col(c).isNotNull() & (F.length(F.trim(F.col(c))) > 0)).cast("int")).alias("nonempty_rows"),
            F.avg(f"{c}_len").alias("avg_length"),
            F.avg(f"{c}_word_count").alias("avg_word_count")
        ).show(truncate=False)

# -----------------------------
# 18. HELPER FUNCTION FOR GROUPED SALARY EDA
# -----------------------------
def salary_group_summary(df, group_col, min_count=20):
    return (
        df.where(
            F.col("annual_salary_eda").isNotNull() &
            (F.col("annual_salary_eda") > 0) &
            F.col(group_col).isNotNull()
        )
        .groupBy(group_col)
        .agg(
            F.count("*").alias("n"),
            F.avg("annual_salary_eda").alias("avg_salary"),
            F.expr("percentile_approx(annual_salary_eda, 0.5)").alias("median_salary")
        )
        .where(F.col("n") >= min_count)
        .orderBy(F.desc("median_salary"))
    )

# -----------------------------
# 19. SALARY VS MAIN POSTING FEATURES
# -----------------------------
print("\n=== Salary vs Main Posting Features ===")
group_cols = [
    "formatted_experience_level",
    "formatted_work_type",
    "work_type",
    "remote_allowed",
    "application_type",
    "compensation_type",
    "currency",
    "pay_period",
    "location"
]

for gc in group_cols:
    if gc in postings_df.columns:
        print(f"\n--- Salary by {gc} ---")
        salary_group_summary(postings_df, gc, min_count=20).show(30, truncate=False)

# -----------------------------
# 20. SALARY BY TITLE
# -----------------------------
print("\n=== Salary by Title ===")
if "title" in postings_df.columns:
    postings_df.where(
        F.col("annual_salary_eda").isNotNull() &
        (F.col("annual_salary_eda") > 0) &
        F.col("title").isNotNull()
    ).groupBy("title").agg(
        F.count("*").alias("n"),
        F.expr("percentile_approx(annual_salary_eda, 0.5)").alias("median_salary")
    ).where(F.col("n") >= 15) \
     .orderBy(F.desc("median_salary")) \
     .show(50, truncate=False)

# -----------------------------
# 21. AGGREGATE BENEFITS BEFORE JOINING
# -----------------------------
# benefits.csv is one-to-many by job_id.
# We aggregate first to avoid duplicating job rows.
print("\n=== Benefits Aggregation ===")
benefits_agg = benefits_df.groupBy("job_id").agg(
    F.count("*").alias("benefit_count"),
    F.countDistinct("type").alias("benefit_type_count"),
    F.max(F.col("inferred").cast("int")).alias("has_inferred_benefit")
)
benefits_agg.show(10, truncate=False)

top_benefits = (
    benefits_df.where(F.col("type").isNotNull())
    .groupBy("type").count()
    .orderBy(F.desc("count"))
    .limit(10)
)
top_benefit_list = [r["type"] for r in top_benefits.collect() if r["type"] is not None]

if top_benefit_list:
    benefits_pivot = (
        benefits_df.where(F.col("type").isin(top_benefit_list))
        .groupBy("job_id")
        .pivot("type", top_benefit_list)
        .agg(F.lit(1))
        .fillna(0)
    )
else:
    benefits_pivot = benefits_agg.select("job_id")

# -----------------------------
# 22. KEEP ONLY LATEST EMPLOYEE COUNT SNAPSHOT
# -----------------------------
# employee_counts may contain multiple time snapshots per company.
# We keep the latest one to avoid duplicate company rows.
print("\n=== Latest Employee Count Snapshot ===")
if "time_recorded" in employee_counts_df.columns:
    employee_window = Window.partitionBy("company_id").orderBy(F.desc("time_recorded"))
    employee_latest = (
        employee_counts_df
        .withColumn("rn", F.row_number().over(employee_window))
        .where(F.col("rn") == 1)
        .drop("rn")
    )
else:
    employee_latest = employee_counts_df.dropDuplicates(["company_id"])

employee_latest.show(10, truncate=False)

# -----------------------------
# 23. BUILD JOINED EDA TABLE
# -----------------------------
#  only join prepared tables.
print("\n=== Build Joined EDA Table ===")
eda_joined = (
    postings_df
    .join(benefits_agg, on="job_id", how="left")
    .join(benefits_pivot, on="job_id", how="left")
    .join(
        companies_df.select(
            *[c for c in ["company_id", "name", "company_size", "country", "state", "city"] if c in companies_df.columns]
        ),
        on="company_id",
        how="left"
    )
    .join(
        employee_latest.select(
            *[c for c in ["company_id", "employee_count", "follower_count"] if c in employee_latest.columns]
        ),
        on="company_id",
        how="left"
    )
    .fillna({
        "benefit_count": 0,
        "benefit_type_count": 0,
        "has_inferred_benefit": 0
    })
)

# Fill top benefit columns with 0 as well
for c in top_benefit_list:
    if c in eda_joined.columns:
        eda_joined = eda_joined.fillna({c: 0})

print("Joined row count:", eda_joined.count())

# Check whether joins caused accidental row duplication.
if "job_id" in eda_joined.columns:
    dup_jobs = eda_joined.groupBy("job_id").count().where(F.col("count") > 1).count()
    print("Duplicate job_id rows after joins:", dup_jobs)

# -----------------------------
# 24. COMPANY / BENEFIT EDA
# -----------------------------
print("\n=== Salary by Company Features ===")
for gc in ["company_size", "country", "state", "city"]:
    if gc in eda_joined.columns:
        print(f"\n--- Salary by {gc} ---")
        salary_group_summary(eda_joined, gc, min_count=20).show(30, truncate=False)

print("\n=== Salary by Benefit Count ===")
if "benefit_count" in eda_joined.columns:
    eda_joined.where(F.col("annual_salary_eda").isNotNull() & (F.col("annual_salary_eda") > 0)) \
        .groupBy("benefit_count") \
        .agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(annual_salary_eda, 0.5)").alias("median_salary")
        ) \
        .where(F.col("n") >= 20) \
        .orderBy("benefit_count") \
        .show(50, truncate=False)



# -----------------------------
# 25. EMPLOYEE COUNT / FOLLOWER COUNT EDA
# -----------------------------
print("\n=== Employee Count / Follower Count Summary ===")
for c in ["employee_count", "follower_count"]:
    if c in eda_joined.columns:
        eda_joined.select(
            F.expr(f"percentile_approx({c}, array(0.25,0.5,0.75,0.95,0.99))").alias(f"{c}_quantiles")
        ).show(truncate=False)

# Bucket employee_count for easier salary analysis
if "employee_count" in eda_joined.columns:
    eda_joined = eda_joined.withColumn(
        "employee_count_bucket",
        F.when(F.col("employee_count").isNull(), "missing")
         .when(F.col("employee_count") < 50, "<50")
         .when(F.col("employee_count") < 200, "50-199")
         .when(F.col("employee_count") < 1000, "200-999")
         .when(F.col("employee_count") < 5000, "1000-4999")
         .otherwise("5000+")
    )

    print("\n=== Salary by Employee Count Bucket ===")
    salary_group_summary(eda_joined, "employee_count_bucket", min_count=20).show(30, truncate=False)

# -----------------------------
# 26. TIME-BASED EDA
# -----------------------------
print("\n=== Time-Based EDA ===")
time_source = None
if "listed_time" in eda_joined.columns:
    time_source = "listed_time"
elif "original_listed_time" in eda_joined.columns:
    time_source = "original_listed_time"

if time_source is not None:
    eda_joined = (
        eda_joined
        .withColumn("listed_year", F.year(F.col(time_source)))
        .withColumn("listed_month", F.month(F.col(time_source)))
        .withColumn("listed_dayofweek", F.dayofweek(F.col(time_source)))
    )

    for gc in ["listed_year", "listed_month", "listed_dayofweek"]:
        print(f"\n--- Salary by {gc} ---")
        salary_group_summary(eda_joined, gc, min_count=20).show(30, truncate=False)

# -----------------------------
# 27. TOP BENEFIT FLAG EDA
# -----------------------------
# If top benefit indicator columns exist, inspect
if top_benefit_list:
    print("\n=== Salary by Top Benefit Flags ===")
    for benefit_col in top_benefit_list:
        if benefit_col in eda_joined.columns:
            print(f"\n--- Salary by benefit flag: {benefit_col} ---")
            salary_group_summary(eda_joined, benefit_col, min_count=20).show(10, truncate=False)

# -----------------------------
# 28. FINAL USABLE SUBSET SUMMARY
# -----------------------------
print("\n=== Final Salary-Usable Subset Summary ===")
salary_usable_df = eda_joined.where(
    F.col("annual_salary_eda").isNotNull() &
    (F.col("annual_salary_eda") > 0)
)

salary_usable_df.select(
    F.count("*").alias("salary_usable_rows"),
    F.countDistinct("job_id").alias("distinct_jobs"),
    F.countDistinct("company_id").alias("distinct_companies")
).show(truncate=False)

if "currency" in salary_usable_df.columns:
    print("\nCurrency distribution in salary-usable subset:")
    salary_usable_df.groupBy("currency").count().orderBy(F.desc("count")).show(20, truncate=False)

if "pay_period" in salary_usable_df.columns:
    print("\nPay period distribution in salary-usable subset:")
    salary_usable_df.groupBy("pay_period").count().orderBy(F.desc("count")).show(20, truncate=False)

# -----------------------------
# 29. FINAL PREVIEW
# -----------------------------
print("\n=== Final Preview of Salary-EDA Table ===")
preview_cols = [c for c in [
    "job_id", "company_id", "title", "location", "currency", "pay_period",
    "min_salary", "med_salary", "max_salary", "normalized_salary_candidate",
    "annual_salary_from_raw", "annual_salary_eda",
    "formatted_experience_level", "formatted_work_type", "remote_allowed",
    "benefit_count", "benefit_type_count", "company_size", "employee_count"
] if c in eda_joined.columns]

eda_joined.select(*preview_cols).show(20, truncate=False)

print("\n=== EDA COMPLETE ===")


=== Raw Table Shapes ===
postings rows: 123849 cols: 31
benefits rows: 67943 cols: 3
companies rows: 24473 cols: 10
employee_counts rows: 35787 cols: 4

=== Cleaned postings schema ===
root
 |-- job_id: long (nullable = true)
 |-- company_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- max_salary: double (nullable = true)
 |-- pay_period: string (nullable = true)
 |-- location: string (nullable = true)
 |-- company_id: long (nullable = true)
 |-- views: long (nullable = true)
 |-- med_salary: double (nullable = true)
 |-- min_salary: double (nullable = true)
 |-- formatted_work_type: string (nullable = true)
 |-- applies: long (nullable = true)
 |-- original_listed_time: timestamp (nullable = true)
 |-- remote_allowed: boolean (nullable = true)
 |-- job_posting_url: string (nullable = true)
 |-- application_url: string (nullable = true)
 |-- application_type: string (nullable = true)
 |-- expiry: timestamp (nullable = 

## 3. Outlier Treatment & Robust Salary Analysis
The initial EDA revealed extreme salary outliers (e.g., test postings with $100M+ salaries) that heavily distort mean calculations.

In this step, we:
* Filter for rows with valid, positive salaries.
* Calculate the 1st and 99th percentiles.
* Create trimmed, winsorized, and log-transformed salary columns to evaluate whether the patterns seen in EDA are genuine or just driven by a few massive outliers.

In [ ]:
# =========================================================
# EXTRA EDA STEP: TRIMMED / WINSORIZED / LOG-SALARY ANALYSIS
# =========================================================
# earlier EDA showed extreme salary outliers.
# - Raw mean salary by group is getting badly distorted.
# - This step creates:
#     1) a trimmed salary subset
#     2) a winsorized salary column
#     3) grouped summaries using:
#           - original salary
#           - trimmed salary
#           - winsorized salary
#           - log salary
# - This helps decide whether the relationships are real or just driven by outliers.
# =========================================================

from pyspark.sql import functions as F

print("\n=== EXTRA EDA: Trimmed / Winsorized / Log-Salary Analysis ===")

# ---------------------------------------------------------
# 1. Start from the joined EDA table and keep only valid salary rows
# ---------------------------------------------------------
robust_df = eda_joined.where(
    F.col("annual_salary_eda").isNotNull() &
    (F.col("annual_salary_eda") > 0)
)

print("Rows with valid positive salary:")
print(robust_df.count())

# ---------------------------------------------------------
# 2. Compute robust salary cutoffs
# ---------------------------------------------------------
# We use 1st and 99th percentiles as default trimming bounds.
# This is not the only possible choice, but it is a strong,
# reasonable next-step check after your main EDA.
bounds = robust_df.select(
    F.expr("percentile_approx(annual_salary_eda, 0.01)").alias("p01"),
    F.expr("percentile_approx(annual_salary_eda, 0.99)").alias("p99"),
    F.expr("percentile_approx(annual_salary_eda, 0.005)").alias("p005"),
    F.expr("percentile_approx(annual_salary_eda, 0.995)").alias("p995"),
    F.expr("percentile_approx(annual_salary_eda, 0.5)").alias("p50")
).collect()[0]

p01 = bounds["p01"]
p99 = bounds["p99"]
p005 = bounds["p005"]
p995 = bounds["p995"]
p50 = bounds["p50"]

print(f"p0.5% = {p005}")
print(f"p1%   = {p01}")
print(f"p50%  = {p50}")
print(f"p99%  = {p99}")
print(f"p99.5%= {p995}")

# ---------------------------------------------------------
# 3. Create trimmed, winsorized, and log salary variants
# ---------------------------------------------------------
# trimmed_salary_keep:
#   only keeps salary values inside [p01, p99]
# winsorized_salary:
#   caps everything below p01 to p01 and above p99 to p99
# log salary versions:
#   useful because salary is heavily right-skewed
robust_df = (
    robust_df
    .withColumn(
        "trimmed_salary_keep",
        F.when(
            (F.col("annual_salary_eda") >= F.lit(p01)) &
            (F.col("annual_salary_eda") <= F.lit(p99)),
            F.col("annual_salary_eda")
        )
    )
    .withColumn(
        "winsorized_salary",
        F.when(F.col("annual_salary_eda") < F.lit(p01), F.lit(p01))
         .when(F.col("annual_salary_eda") > F.lit(p99), F.lit(p99))
         .otherwise(F.col("annual_salary_eda"))
    )
    .withColumn(
        "log_salary",
        F.log1p(F.col("annual_salary_eda"))
    )
    .withColumn(
        "log_winsorized_salary",
        F.log1p(F.col("winsorized_salary"))
    )
)

print("\n=== Row counts after trimming ===")
robust_df.select(
    F.count("*").alias("all_valid_salary_rows"),
    F.sum(F.col("trimmed_salary_keep").isNotNull().cast("int")).alias("rows_after_1pct_trim")
).show(truncate=False)

# ---------------------------------------------------------
# 4. Compare overall distributions before vs after robust handling
# ---------------------------------------------------------
print("\n=== Overall salary comparison: original vs winsorized vs trimmed ===")
robust_df.select(
    F.min("annual_salary_eda").alias("orig_min"),
    F.expr("percentile_approx(annual_salary_eda, 0.5)").alias("orig_median"),
    F.avg("annual_salary_eda").alias("orig_mean"),
    F.max("annual_salary_eda").alias("orig_max"),

    F.min("winsorized_salary").alias("win_min"),
    F.expr("percentile_approx(winsorized_salary, 0.5)").alias("win_median"),
    F.avg("winsorized_salary").alias("win_mean"),
    F.max("winsorized_salary").alias("win_max"),

    F.min("trimmed_salary_keep").alias("trim_min"),
    F.expr("percentile_approx(trimmed_salary_keep, 0.5)").alias("trim_median"),
    F.avg("trimmed_salary_keep").alias("trim_mean"),
    F.max("trimmed_salary_keep").alias("trim_max")
).show(truncate=False)

# ---------------------------------------------------------
# 5. Helper: grouped summary using multiple salary views
# ---------------------------------------------------------
# This lets you see whether a pattern is stable across:
# - raw salary
# - trimmed salary
# - winsorized salary
# - log salary
#
# If a result disappears after trimming, it was probably driven by outliers.
def robust_salary_group_summary(df, group_col, min_count=20):
    return (
        df.where(F.col(group_col).isNotNull())
          .groupBy(group_col)
          .agg(
              F.count("*").alias("n"),

              F.expr("percentile_approx(annual_salary_eda, 0.5)").alias("raw_median"),
              F.avg("annual_salary_eda").alias("raw_mean"),

              F.expr("percentile_approx(trimmed_salary_keep, 0.5)").alias("trim_median"),
              F.avg("trimmed_salary_keep").alias("trim_mean"),

              F.expr("percentile_approx(winsorized_salary, 0.5)").alias("win_median"),
              F.avg("winsorized_salary").alias("win_mean"),

              F.avg("log_salary").alias("avg_log_salary"),
              F.avg("log_winsorized_salary").alias("avg_log_winsorized_salary")
          )
          .where(F.col("n") >= min_count)
          .orderBy(F.desc("trim_median"))
    )

# ---------------------------------------------------------
# 6. Re-run the main grouped salary EDA with robust targets
# ---------------------------------------------------------
# These are the most important fields to re-check after outlier control.
robust_group_cols = [
    "formatted_experience_level",
    "formatted_work_type",
    "work_type",
    "remote_allowed",
    "application_type",
    "company_size",
    "employee_count_bucket",
    "country",
    "state"
]

for gc in robust_group_cols:
    if gc in robust_df.columns:
        print(f"\n=== Robust salary summary by {gc} ===")
        robust_salary_group_summary(robust_df, gc, min_count=20).show(30, truncate=False)

# ---------------------------------------------------------
# 7. Re-run salary by pay period and currency
# ---------------------------------------------------------
# This checks whether one pay period or weird currency rows
# are still driving strange patterns even after normalization.
for gc in ["pay_period", "currency"]:
    if gc in robust_df.columns:
        print(f"\n=== Robust salary summary by {gc} ===")
        robust_salary_group_summary(robust_df, gc, min_count=5).show(30, truncate=False)

# ---------------------------------------------------------
# 8. Re-run salary by top titles using robust targets
# ---------------------------------------------------------
# Titles are strong predictors, but they were also vulnerable
# to distortion from a few huge outliers.
if "title" in robust_df.columns:
    print("\n=== Robust salary by title ===")
    robust_salary_group_summary(robust_df, "title", min_count=15).show(50, truncate=False)

# ---------------------------------------------------------
# 9. Re-run salary by benefit signals
# ---------------------------------------------------------
if "benefit_count" in robust_df.columns:
    print("\n=== Robust salary by benefit_count ===")
    robust_salary_group_summary(robust_df, "benefit_count", min_count=20).show(50, truncate=False)

if "benefit_type_count" in robust_df.columns:
    print("\n=== Robust salary by benefit_type_count ===")
    robust_salary_group_summary(robust_df, "benefit_type_count", min_count=20).show(50, truncate=False)

# ---------------------------------------------------------
# 10. Optional: inspect the rows that were trimmed out
# ---------------------------------------------------------
# This helps you see whether the trimmed rows are obviously bogus
# or whether some are real but extreme.
print("\n=== Example rows removed by 1%-99% trimming ===")
trimmed_out_df = robust_df.where(F.col("trimmed_salary_keep").isNull())

trimmed_preview_cols = [
    c for c in [
        "job_id", "title", "location", "currency", "pay_period",
        "annual_salary_eda", "normalized_salary_candidate",
        "annual_salary_from_raw", "formatted_experience_level",
        "formatted_work_type"
    ] if c in trimmed_out_df.columns
]

trimmed_out_df.select(*trimmed_preview_cols) \
    .orderBy(F.desc("annual_salary_eda")) \
    .show(40, truncate=False)




=== EXTRA EDA: Trimmed / Winsorized / Log-Salary Analysis ===
Rows with valid positive salary:
36059
p0.5% = 38.68
p1%   = 145.0
p50%  = 81500.0
p99%  = 300000.0
p99.5%= 393500.0

=== Row counts after trimming ===
+---------------------+--------------------+
|all_valid_salary_rows|rows_after_1pct_trim|
+---------------------+--------------------+
|36059                |35338               |
+---------------------+--------------------+


=== Overall salary comparison: original vs winsorized vs trimmed ===
+--------+-----------+-----------------+--------+-------+----------+-----------------+--------+--------+-----------+-----------------+--------+
|orig_min|orig_median|orig_mean        |orig_max|win_min|win_median|win_mean         |win_max |trim_min|trim_median|trim_mean        |trim_max|
+--------+-----------+-----------------+--------+-------+----------+-----------------+--------+--------+-----------+-----------------+--------+
|1.0     |81500.0    |205406.7556026231|5.356E8 |145.0  |

## 4. Base Modeling Dataset Construction
With the EDA and outlier analysis complete, we now construct a clean, one-row-per-job dataset ready for downstream machine learning.

**Steps included:**
* Filtering strictly for USD roles within our 1st-99th percentile trim bounds.
* Filling nulls in categorical columns with explicit "unknown" labels.
* Extracting basic text features (lengths, word counts) and regex-based flag features (e.g., `flag_senior`, `desc_has_python`).
* Dropping leaky, redundant, or extremely sparse columns.

In [ ]:
# ============================================================
# FINAL MODELING DATASET BUILDER
# Creates a clean, feature-engineered, one-row-per-job dataset
# for downstream ML salary prediction.
#
# Key decisions based on EDA:
# - Use annual_salary_eda as target
# - Restrict to USD
# - Keep only positive salary rows
# - Trim salary to the 1st-99th percentile range
# - Drop obviously weak/leaky/useless fields
# - Keep structured + engineered features
# - Save final dataset to CSV and Parquet
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 0. CHECK BASE TABLE
# ------------------------------------------------------------
# Assumes eda_joined already exists from your EDA steps.
# That table should already be:
# - one row per job
# - joined safely with benefits / companies / employee counts
# - include annual_salary_eda
# ------------------------------------------------------------
base_df = eda_joined

print("Base rows:", base_df.count())
print("Base cols:", len(base_df.columns))

# ------------------------------------------------------------
# 1. KEEP ONLY CLEAN TARGET ROWS
# ------------------------------------------------------------
# Based on EDA:
# - annual_salary_eda is the usable annualized target
# - USD dominates the salary-usable subset
# - extreme salary rows should be removed, not trusted
# ------------------------------------------------------------
target_df = base_df.where(
    F.col("annual_salary_eda").isNotNull() &
    (F.col("annual_salary_eda") > 0) &
    (F.col("currency") == "USD")
)

print("\nRows after requiring positive USD salary:")
print(target_df.count())

# ------------------------------------------------------------
# 2. COMPUTE TRIMMING BOUNDS
# ------------------------------------------------------------
# EDA showed 1%-99% trim is a sensible rule here.
# We use it to remove obvious garbage salaries.
# ------------------------------------------------------------
bounds = target_df.select(
    F.expr("percentile_approx(annual_salary_eda, 0.01)").alias("p01"),
    F.expr("percentile_approx(annual_salary_eda, 0.99)").alias("p99")
).collect()[0]

p01 = bounds["p01"]
p99 = bounds["p99"]

print("\nTrim bounds:")
print("p01 =", p01)
print("p99 =", p99)

target_df = target_df.where(
    (F.col("annual_salary_eda") >= F.lit(p01)) &
    (F.col("annual_salary_eda") <= F.lit(p99))
)

print("\nRows after 1%-99% trim:")
print(target_df.count())

# ------------------------------------------------------------
# 3. ADD ROBUST TARGET VERSIONS
# ------------------------------------------------------------
# Keep both raw annual salary and log salary.
# The log target is likely better for ML.
# ------------------------------------------------------------
target_df = (
    target_df
    .withColumn("target_salary", F.col("annual_salary_eda"))
    .withColumn("target_log_salary", F.log1p(F.col("annual_salary_eda")))
)

# ------------------------------------------------------------
# 4. CLEAN / STANDARDIZE KEY CATEGORICALS
# ------------------------------------------------------------
# Use explicit 'unknown' categories instead of dropping nulls.
# This is important because several columns are structurally sparse.
# ------------------------------------------------------------
categorical_fill_map = {}

for c in [
    "formatted_experience_level",
    "formatted_work_type",
    "work_type",
    "application_type",
    "company_size",
    "country",
    "state",
    "city",
    "posting_domain"
]:
    if c in target_df.columns:
        target_df = target_df.withColumn(c, F.trim(F.col(c).cast("string")))
        categorical_fill_map[c] = "unknown"

# remote_allowed is especially sparse, so force it into a 3-level categorical
if "remote_allowed" in target_df.columns:
    target_df = target_df.withColumn(
        "remote_allowed_filled",
        F.when(F.col("remote_allowed") == True, F.lit("true"))
         .when(F.col("remote_allowed") == False, F.lit("false"))
         .otherwise(F.lit("unknown"))
    )

# fill other categorical columns
if categorical_fill_map:
    target_df = target_df.fillna(categorical_fill_map)

# ------------------------------------------------------------
# 5. TEXT CLEANING + BASIC TEXT FEATURES
# ------------------------------------------------------------
# Based on EDA:
# - title and description are strong
# - skills_desc is too sparse, so exclude it from core model
#
# We keep:
# - cleaned title text
# - cleaned description text
# - title/description length features
# - word count features
# - keyword / seniority / role-family indicators
# ------------------------------------------------------------
for c in ["title", "description", "location"]:
    if c in target_df.columns:
        target_df = target_df.withColumn(
            f"{c}_clean",
            F.lower(F.trim(F.regexp_replace(F.coalesce(F.col(c), F.lit("")), r"\s+", " ")))
        )

if "title_clean" in target_df.columns:
    target_df = target_df.withColumn("title_len", F.length(F.col("title_clean")))
    target_df = target_df.withColumn("title_word_count", F.size(F.split(F.col("title_clean"), r"\s+")))

if "description_clean" in target_df.columns:
    target_df = target_df.withColumn("description_len", F.length(F.col("description_clean")))
    target_df = target_df.withColumn("description_word_count", F.size(F.split(F.col("description_clean"), r"\s+")))

# Seniority/title flags
if "title_clean" in target_df.columns:
    target_df = (
        target_df
        .withColumn("flag_senior", F.col("title_clean").rlike(r"\bsenior\b|\bsr\b"))
        .withColumn("flag_lead", F.col("title_clean").rlike(r"\blead\b"))
        .withColumn("flag_manager", F.col("title_clean").rlike(r"\bmanager\b"))
        .withColumn("flag_director", F.col("title_clean").rlike(r"\bdirector\b"))
        .withColumn("flag_executive", F.col("title_clean").rlike(r"\bchief\b|\bcfo\b|\bceo\b|\bcoo\b|\bcto\b|\bpresident\b|\bvice president\b|\bvp\b"))
        .withColumn("flag_intern", F.col("title_clean").rlike(r"\bintern\b|\binternship\b"))
        .withColumn("flag_associate", F.col("title_clean").rlike(r"\bassociate\b"))
        .withColumn("flag_engineer", F.col("title_clean").rlike(r"\bengineer\b"))
        .withColumn("flag_data", F.col("title_clean").rlike(r"\bdata\b|\banalyst\b|\bscientist\b|\bmachine learning\b|\bml\b"))
        .withColumn("flag_sales", F.col("title_clean").rlike(r"\bsales\b|\bbusiness development\b|\baccount executive\b"))
        .withColumn("flag_finance", F.col("title_clean").rlike(r"\bfinance\b|\bfinancial\b|\baccounting\b|\bcontroller\b|\battorney\b|\btax\b"))
        .withColumn("flag_healthcare", F.col("title_clean").rlike(r"\bnurse\b|\bphysician\b|\bmedical\b|\bclinical\b|\bdental\b|\bveterinarian\b"))
        .withColumn("flag_project_product", F.col("title_clean").rlike(r"\bproject manager\b|\bprogram manager\b|\bproduct manager\b"))
    )

# Description keyword flags
if "description_clean" in target_df.columns:
    target_df = (
        target_df
        .withColumn("desc_has_python", F.col("description_clean").rlike(r"\bpython\b"))
        .withColumn("desc_has_sql", F.col("description_clean").rlike(r"\bsql\b"))
        .withColumn("desc_has_java", F.col("description_clean").rlike(r"\bjava\b"))
        .withColumn("desc_has_cloud", F.col("description_clean").rlike(r"\baws\b|\bazure\b|\bgcp\b|\bcloud\b"))
        .withColumn("desc_has_ai_ml", F.col("description_clean").rlike(r"\bmachine learning\b|\bartificial intelligence\b|\bdeep learning\b|\bai\b"))
        .withColumn("desc_has_management", F.col("description_clean").rlike(r"\bmanage\b|\bmanagement\b|\bleadership\b"))
        .withColumn("desc_has_degree", F.col("description_clean").rlike(r"\bbachelor\b|\bmaster\b|\bphd\b|\bdegree\b"))
        .withColumn("desc_has_certification", F.col("description_clean").rlike(r"\bcertification\b|\blicense\b|\blicensed\b"))
    )

# ------------------------------------------------------------
# 6. TIME FEATURES
# ------------------------------------------------------------
# EDA said time signal is not the main story, but these are cheap
# and harmless features if parsed timestamps exist.
# ------------------------------------------------------------
time_source = None
if "listed_time" in target_df.columns:
    time_source = "listed_time"
elif "original_listed_time" in target_df.columns:
    time_source = "original_listed_time"

if time_source is not None:
    target_df = (
        target_df
        .withColumn("listed_year", F.year(F.col(time_source)))
        .withColumn("listed_month", F.month(F.col(time_source)))
        .withColumn("listed_dayofweek", F.dayofweek(F.col(time_source)))
    )

# ------------------------------------------------------------
# 7. BENEFITS FEATURES
# ------------------------------------------------------------
# Keep benefit count features and a few top benefit indicators if present.
# These survived EDA as mildly useful signals.
# ------------------------------------------------------------
for c in ["benefit_count", "benefit_type_count", "has_inferred_benefit"]:
    if c in target_df.columns:
        target_df = target_df.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(0.0)))

# Keep any top-benefit pivot columns if they are already 0/1 numeric.
# We infer them as columns not in the original base posting set and with low distinct count.
# Simpler and safer: explicitly keep 0/1 benefit pivots if they are integer-like columns
# and not obvious IDs or targets. We do that later in final column selection.

# ------------------------------------------------------------
# 8. COMPANY / SCALE FEATURES
# ------------------------------------------------------------
# Company features are sparse but still useful.
# Add missing flags + log transforms for skewed counts.
# ------------------------------------------------------------
for c in ["employee_count", "follower_count"]:
    if c in target_df.columns:
        target_df = target_df.withColumn(c, F.col(c).cast("double"))

if "employee_count" in target_df.columns:
    target_df = (
        target_df
        .withColumn("employee_count_missing", F.col("employee_count").isNull().cast("int"))
        .withColumn("employee_count_filled", F.coalesce(F.col("employee_count"), F.lit(0.0)))
        .withColumn("log_employee_count", F.log1p(F.col("employee_count_filled")))
    )

if "follower_count" in target_df.columns:
    target_df = (
        target_df
        .withColumn("follower_count_missing", F.col("follower_count").isNull().cast("int"))
        .withColumn("follower_count_filled", F.coalesce(F.col("follower_count"), F.lit(0.0)))
        .withColumn("log_follower_count", F.log1p(F.col("follower_count_filled")))
    )

# employee_count_bucket was useful in EDA; recreate if missing
if "employee_count_bucket" not in target_df.columns and "employee_count" in target_df.columns:
    target_df = target_df.withColumn(
        "employee_count_bucket",
        F.when(F.col("employee_count").isNull(), "missing")
         .when(F.col("employee_count") < 50, "<50")
         .when(F.col("employee_count") < 200, "50-199")
         .when(F.col("employee_count") < 1000, "200-999")
         .when(F.col("employee_count") < 5000, "1000-4999")
         .otherwise("5000+")
    )

# ------------------------------------------------------------
# 9. LOCATION FEATURES
# ------------------------------------------------------------
# Keep raw location text and simple engineered flags.
# Location clearly matters, but raw posting location can be messy.
# ------------------------------------------------------------
if "location_clean" in target_df.columns:
    target_df = (
        target_df
        .withColumn("location_word_count", F.size(F.split(F.col("location_clean"), r"\s+")))
        .withColumn("location_has_remote_word", F.col("location_clean").rlike(r"\bremote\b|\bhybrid\b"))
    )

# ------------------------------------------------------------
# 10. LOW-QUALITY / LEAKY / USELESS COLUMN DROPS
# ------------------------------------------------------------
# Drop columns that should not go into the final ML CSV.
# We keep target columns and useful engineered features only.
# ------------------------------------------------------------
drop_cols = [
    # IDs / URLs / raw identifiers
    "job_id",
    "company_id",
    "job_posting_url",
    "application_url",
    "url",
    "address",
    "zip_code",
    "fips",

    # raw sparse / weak text
    "skills_desc",

    # raw salary ingredients (not needed as predictors once target is built)
    "min_salary",
    "med_salary",
    "max_salary",
    "normalized_salary",
    "normalized_salary_candidate",
    "range_mid_salary",
    "salary_target_raw",
    "annual_salary_from_raw",
    "annual_salary_eda",
    "log_annual_salary_eda",

    # post-listing or likely leakage-like platform fields / low priority
    "closed_time",
    "expiry",

    # weak / potentially leakage-like engagement metrics
    "views",
    "applies",

    # raw booleans replaced by cleaned categorical versions
    "remote_allowed",

    # raw text versions replaced by cleaned versions
    "title",
    "description",
    "location",

    # raw count versions replaced by filled/log versions
    "employee_count",
    "follower_count",
]

existing_drop_cols = [c for c in drop_cols if c in target_df.columns]
final_df = target_df.drop(*existing_drop_cols)

# ------------------------------------------------------------
# 11. FINAL COLUMN SELECTION
# ------------------------------------------------------------
# Keep only columns that are likely useful for ML.
# This avoids passing through lots of junk from the raw joins.
# ------------------------------------------------------------
preferred_cols = [
    # targets
    "target_salary",
    "target_log_salary",

    # cleaned text
    "title_clean",
    "description_clean",
    "location_clean",

    # core categoricals
    "formatted_experience_level",
    "formatted_work_type",
    "work_type",
    "application_type",
    "compensation_type",
    "remote_allowed_filled",
    "company_size",
    "country",
    "state",
    "city",
    "posting_domain",
    "employee_count_bucket",

    # text lengths
    "title_len",
    "title_word_count",
    "description_len",
    "description_word_count",
    "location_word_count",

    # title flags
    "flag_senior",
    "flag_lead",
    "flag_manager",
    "flag_director",
    "flag_executive",
    "flag_intern",
    "flag_associate",
    "flag_engineer",
    "flag_data",
    "flag_sales",
    "flag_finance",
    "flag_healthcare",
    "flag_project_product",

    # description flags
    "desc_has_python",
    "desc_has_sql",
    "desc_has_java",
    "desc_has_cloud",
    "desc_has_ai_ml",
    "desc_has_management",
    "desc_has_degree",
    "desc_has_certification",

    # benefits
    "benefit_count",
    "benefit_type_count",
    "has_inferred_benefit",

    # company scale
    "employee_count_missing",
    "employee_count_filled",
    "log_employee_count",
    "follower_count_missing",
    "follower_count_filled",
    "log_follower_count",

    # simple location/time
    "location_has_remote_word",
    "listed_year",
    "listed_month",
    "listed_dayofweek",
]

# Also keep any 0/1 top-benefit pivot columns automatically
auto_keep_cols = []
for c, dtype in final_df.dtypes:
    if c in preferred_cols:
        continue
    if c.startswith("benefit_"):
        continue
    # keep columns that look like top-benefit pivots:
    # numeric, low-distinct, and not obviously junk
    if dtype in ("int", "bigint", "double", "float"):
        if c not in ["target_salary", "target_log_salary"]:
            auto_keep_cols.append(c)

# To avoid keeping too much junk, filter auto_keep_cols heavily:
blacklist_auto = {
    "sponsored",
    "has_inferred_benefit",
    "listed_year",
    "listed_month",
    "listed_dayofweek",
    "benefit_count",
    "benefit_type_count",
    "employee_count_filled",
    "follower_count_filled",
    "title_len",
    "title_word_count",
    "description_len",
    "description_word_count",
    "location_word_count",
}
auto_keep_cols = [c for c in auto_keep_cols if c not in blacklist_auto]

# Keep only existing columns
final_cols = [c for c in preferred_cols if c in final_df.columns]

# Also add safe top-benefit pivots if they are binary-like and not already included
for c in auto_keep_cols:
    if c not in final_cols:
        # quick guard: don't accidentally keep weird numeric junk
        if c not in ["salary_abs_diff", "salary_pct_diff"]:
            final_cols.append(c)

# Final selected table
final_df = final_df.select(*final_cols)

# ------------------------------------------------------------
# 12. FINAL CLEANUP OF NULLS FOR CSV OUTPUT
# ------------------------------------------------------------
# CSV is easier to use later if obvious nulls are standardized.
# ------------------------------------------------------------
# Fill booleans / flags with 0 when null
flag_like_cols = [
    c for c in final_df.columns
    if c.startswith("flag_") or c.startswith("desc_has_") or c.endswith("_missing") or c.endswith("_word")
]

for c in final_df.columns:
    if c.startswith("flag_") or c.startswith("desc_has_"):
        final_df = final_df.withColumn(c, F.coalesce(F.col(c).cast("int"), F.lit(0)))

if "location_has_remote_word" in final_df.columns:
    final_df = final_df.withColumn("location_has_remote_word", F.coalesce(F.col("location_has_remote_word").cast("int"), F.lit(0)))

# Fill numerics with 0 only for engineered count/log features where that makes sense
fill_zero_map = {}
for c in [
    "benefit_count",
    "benefit_type_count",
    "has_inferred_benefit",
    "employee_count_missing",
    "employee_count_filled",
    "log_employee_count",
    "follower_count_missing",
    "follower_count_filled",
    "log_follower_count",
    "title_len",
    "title_word_count",
    "description_len",
    "description_word_count",
    "location_word_count",
    "listed_year",
    "listed_month",
    "listed_dayofweek",
]:
    if c in final_df.columns:
        fill_zero_map[c] = 0

if fill_zero_map:
    final_df = final_df.fillna(fill_zero_map)

# Fill remaining categoricals/text with "unknown" or empty string
fill_unknown_map = {}
for c, dtype in final_df.dtypes:
    if dtype == "string":
        if c in ["title_clean", "description_clean", "location_clean"]:
            fill_unknown_map[c] = ""
        else:
            fill_unknown_map[c] = "unknown"

if fill_unknown_map:
    final_df = final_df.fillna(fill_unknown_map)

# ------------------------------------------------------------
# 13. FINAL VALIDATION
# ------------------------------------------------------------
print("\n=== Final modeling dataset validation ===")
print("Rows:", final_df.count())
print("Columns:", len(final_df.columns))

final_df.select(
    F.count("*").alias("rows"),
    F.sum(F.col("target_salary").isNull().cast("int")).alias("null_target_salary"),
    F.sum(F.col("target_log_salary").isNull().cast("int")).alias("null_target_log_salary"),
    F.min("target_salary").alias("min_target_salary"),
    F.max("target_salary").alias("max_target_salary"),
    F.avg("target_salary").alias("avg_target_salary")
).show(truncate=False)

print("\nFinal schema:")
final_df.printSchema()

print("\nPreview:")
final_df.show(10, truncate=False)


Base rows: 123849
Base cols: 67

Rows after requiring positive USD salary:
36044

Trim bounds:
p01 = 147.5
p99 = 300000.0

Rows after 1%-99% trim:
35321

=== Final modeling dataset validation ===
Rows: 35321
Columns: 68
+-----+------------------+----------------------+-----------------+-----------------+-----------------+
|rows |null_target_salary|null_target_log_salary|min_target_salary|max_target_salary|avg_target_salary|
+-----+------------------+----------------------+-----------------+-----------------+-----------------+
|35321|0                 |0                     |147.5            |300000.0         |93513.06940573598|
+-----+------------------+----------------------+-----------------+-----------------+-----------------+


Final schema:
root
 |-- target_salary: double (nullable = true)
 |-- target_log_salary: double (nullable = true)
 |-- title_clean: string (nullable = false)
 |-- description_clean: string (nullable = false)
 |-- location_clean: string (nullable = false)
 |--

## 5. Feature Engineering
We will build upon our cleaned dataset by creating higher-order features to capture the nuances of a job posting's value.

**Engineered Features:**
* `seniority_score`: An ordinal scale based on title flags (Associate -> Executive).
* `tech_skill_count`: An aggregate count of mentioned technical skills (Python, SQL, Cloud, AI/ML).
* `company_prestige`: A proxy metric multiplying log-transformed employee and follower counts.
* `manager_x_company_size`: An interaction term capturing the scale of a managerial role.
* `exp_level_ordinal`: An ordinal encoding of the required experience level.
* `location_state`: Extracted state codes from the cleaned location strings.

In [ ]:
from pyspark.sql import functions as F
df = final_df
# 1. Seniority ordinal score
df = df.withColumn('seniority_score',
    (F.col('flag_associate') * 1 +
     F.col('flag_senior')    * 2 +
     F.col('flag_lead')      * 3 +
     F.col('flag_manager')   * 3 +
     F.col('flag_director')  * 4).cast('double')
)

# 2. Tech skill count
df = df.withColumn('tech_skill_count',
    (F.col('desc_has_python') + F.col('desc_has_sql') +
     F.col('desc_has_cloud')  + F.col('desc_has_ai_ml')).cast('double')
)

# 3. Company prestige proxy - Proxy with employee x follower count
df = df.withColumn('company_prestige',
    (F.col('log_employee_count') * F.col('log_follower_count')).cast('double')
)


# 4. Interaction: manager × log_employee_count
df = df.withColumn('manager_x_company_size',
    (F.col('flag_manager') * F.col('log_employee_count')).cast('double')
)

# 5. Ordinal encoding for formatted_experience_level

exp_order = {
    'Internship': 0.0,
    'Entry level': 1.0,
    'Associate': 2.0,
    'Mid-Senior level': 3.0,
    'Director': 4.0,
    'Executive': 5.0
}
exp_mapping = F.create_map(*[
    item for pair in [(F.lit(k), F.lit(v)) for k, v in exp_order.items()] for item in pair
])
df = df.withColumn('exp_level_ordinal',
    F.coalesce(exp_mapping[F.col('formatted_experience_level')], F.lit(2.0))  # default: mid
)

# 6. Location state extraction
df = df.withColumn(
    'location_state',
    F.when(
        F.regexp_extract(F.col('location_clean'), r',\s*([a-z]{2})$', 1) != '',
        F.upper(F.regexp_extract(F.col('location_clean'), r',\s*([a-z]{2})$', 1))
    ).otherwise('Other')
)


## 6. LLM-Powered Agentic Feature Selection
To systematically drop weak or redundant features, we deploy an AI Agent using OpenAI and LangGraph.

The agent acts as a Lead Data Scientist, equipped with PySpark tools to calculate missingness, distinct counts, and statistical signals (Pearson correlations for continuous variables, median variances for categoricals). Based on the data it retrieves, the agent autonomously decides which features lack predictive power and marks them for removal.

In [ ]:
import json
import pyspark.sql.functions as F
from pydantic import BaseModel, Field
from typing import List, Dict, Any
import os
import json
from typing import TypedDict, Dict, Any, List

from openai import OpenAI
from langgraph.graph import StateGraph, START, END
from pyspark.sql import functions as F

# -----------------------------
# 1. CONNECT TO OPENAI
# -----------------------------

from google.colab import userdata
YOUR_OPENAI_API_KEY =userdata.get('YOUR_OPENAI_API_KEY')

os.environ["OPENAI_API_KEY"] = YOUR_OPENAI_API_KEY

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set. Set it before running this cell.")

import json
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType
from pydantic import BaseModel, Field
from typing import List, Dict, Any

# ==========================================
# 1. DEFINE PYSPARK SKILLS (FUNCTIONS)
# ==========================================
agent_dropped_columns = []

def evaluate_feature_signal(column_name: str) -> str:
    """
    Agent Skill: Calculates missingness, variance, and signal strength
    against target_log_salary.
    """
    try:
        total_rows = df.count()

        # 1. Get data type FIRST to prevent casting errors
        col_type = dict(df.dtypes).get(column_name, "string")
        is_numeric = col_type in ('int', 'bigint', 'double', 'float', 'long')

        # 2. Calculate Nulls safely based on data type
        if is_numeric:
            null_count = df.filter(F.col(column_name).isNull() | F.isnan(F.col(column_name))).count()
        else:
            null_count = df.filter(F.col(column_name).isNull()).count()

        null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
        distinct_count = df.select(column_name).distinct().count()

        result = {
            "column": column_name,
            "type": col_type,
            "null_percentage": round(null_pct, 2),
            "distinct_values_count": distinct_count,
            "signal_type": "None",
            "signal_data": {}
        }

        # 3. Calculate Signal Strength
        if distinct_count > 1 and null_pct < 100:
            if is_numeric:
                # Pearson Correlation for Numerics
                corr = df.stat.corr(column_name, "target_log_salary")
                result["signal_type"] = "pearson_correlation"
                result["signal_data"] = {"correlation_with_target": round(corr, 4) if corr else 0.0}
            else:
                # Median Target Salary Difference for Categoricals/Booleans
                group_stats = (
                    df.filter(F.col(column_name).isNotNull())
                      .groupBy(column_name)
                      .agg(
                          F.count("*").alias("count"),
                          F.expr("percentile_approx(target_log_salary, 0.5)").alias("median_log_salary")
                      )
                      .orderBy(F.desc("count"))
                      .limit(10)
                      .collect()
                )

                medians = [
                    {
                        "category": str(row[column_name]),
                        "count": row["count"],
                        "median_log_salary": round(row["median_log_salary"], 3) if row["median_log_salary"] else None
                    }
                    for row in group_stats
                ]
                result["signal_type"] = "group_medians"
                result["signal_data"] = {"top_categories": medians}

        return json.dumps(result)
    except Exception as e:
        return json.dumps({"error": str(e)})

def drop_feature(columns_to_drop: List[str], justification: str) -> str:
    """Agent Skill: Marks columns for removal."""
    global agent_dropped_columns
    agent_dropped_columns.extend(columns_to_drop)
    print(f"🤖 Agent dropped {columns_to_drop}. Reason: {justification}")
    return json.dumps({"status": "success", "dropped": columns_to_drop})

def finalize_feature_selection(status: str) -> str:
    """Agent Skill: Signals that the agent has finished evaluating."""
    return json.dumps({"status": "finalized"})

TOOL_FUNCTIONS = {
    "evaluate_feature_signal": evaluate_feature_signal,
    "drop_feature": drop_feature,
    "finalize_feature_selection": finalize_feature_selection
}

# ==========================================
# 2. DEFINE PYDANTIC SCHEMAS & TOOLS
# ==========================================
class EvaluateFeatureArgs(BaseModel):
    column_name: str = Field(..., description="Name of the single column to evaluate.")

class DropFeatureArgs(BaseModel):
    columns_to_drop: List[str] = Field(..., description="List of column names to drop.")
    justification: str = Field(..., description="Statistical reasoning for dropping these columns based on weak correlation or lack of variance across group medians.")

class FinalizeArgs(BaseModel):
    status: str = Field(..., description="Pass 'done' when you have finished evaluating the requested columns.")

def pydantic_to_tool(name: str, description: str, model: type) -> Dict[str, Any]:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": model.model_json_schema()
        }
    }

TOOLS = [
    pydantic_to_tool("evaluate_feature_signal", "Get null percentage, distinct counts, and signal strength (correlation or group medians) against target_log_salary.", EvaluateFeatureArgs),
    pydantic_to_tool("drop_feature", "Add columns to the drop list if they lack statistical signal or have high missingness.", DropFeatureArgs),
    pydantic_to_tool("finalize_feature_selection", "Call this strictly when you have finished evaluating ALL candidate columns.", FinalizeArgs)
]

# ==========================================
# 3. DEFINE THE AGENT LOOP
# ==========================================
def run_feature_selection_agent(candidate_columns: List[str], max_steps: int = 20):
    global agent_dropped_columns
    agent_dropped_columns = []

    system_prompt = (
        "You are a Lead Data Scientist performing feature selection on a PySpark DataFrame. "
        "Your goal is to evaluate the provided candidate columns for predicting 'target_log_salary'.\n"
        "1. Use 'evaluate_feature_signal' to inspect columns.\n"
        "2. If the feature is numeric, look at the pearson correlation. If it is close to 0 (e.g., more than -0.05 and less than 0.05), it is a weak signal.\n"
        "3. If the feature is categorical, look at the group medians. If the median target_log_salary is practically identical across all major groups, it is a weak signal.\n"
        "4. Use 'drop_feature' to remove columns with weak signals, extremely high null rates (> 80%), or no variance (1 distinct value).\n"
        "5. When you have checked all candidate columns, call 'finalize_feature_selection'."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Please evaluate these columns: {candidate_columns}"}
    ]

    print("🚀 Starting Agent Feature Selection Loop...\n")

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )

        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                func_name = tc.function.name
                args = json.loads(tc.function.arguments)

                if func_name == "finalize_feature_selection":
                    print("✅ Agent finished evaluation.")
                    return list(set(agent_dropped_columns))

                print(f"🛠️ Agent calling `{func_name}` with arguments: {args}")
                result = TOOL_FUNCTIONS[func_name](**args)

                messages.append({
                    "tool_call_id": tc.id,
                    "role": "tool",
                    "content": result
                })
        else:
            print(f"Agent says: {msg.content}")
            if "done" in str(msg.content).lower() or "finalized" in str(msg.content).lower():
                 break

    print("⚠️ Reached max steps before finalization.")
    return list(set(agent_dropped_columns))

# ==========================================
# 4. EXECUTE THE WORKFLOW
# ==========================================
# Drop columns that are redundant
redundant_columns_to_drop = [
    # --- TARGET LEAKAGE ---
    # Direct versions of the prediction target
    "target_salary",

    # --- FEATURE ENGINEERING REPLACEMENTS ---
    # Columns replaced by cleaner, ordinal, or derived versions
    "formatted_experience_level",  # Replaced by exp_level_ordinal
    "location_clean",              # Used to derive location_state
    "city",                        # Used location_state instead
    "state",                       # Replaced by location_state
    "work_type",                   # formatted_work_type is used instead

    # --- COMPANY METRICS & DATA QUALITY ---
    # Log-transformed signals or categorical buckets are preferred
    "employee_count_missing",
    "employee_count_filled",
    "follower_count_missing",
    "follower_count_filled",

    # --- NLP & TEXT METRIC REDUNDANCY ---
    # Length/word counts that are redundant with description_len or handled via TF-IDF
    "title_len",
    "title_word_count",
    "description_word_count",
    "skills_desc_len",
    "skills_desc_word_count",
    "location_word_count",

    # --- BENEFIT AGGREGATES ---
    # Overlap with the primary benefit_count
    "benefit_type_count",
    "has_inferred_benefit",

    # --- CATEGORICAL SELECTION ---
    # Dropped Non-Informative Input
    "posting_domain",
]

new_df = df.drop(*redundant_columns_to_drop)

# Columns that may not be relevant
columns_to_evaluate = [
    "benefit_count",
    "description_len",
    "compensation_type",
    "listed_year",
    "listed_month",
    "listed_dayofweek",
    "Paid maternity leave",
    "Paid paternity leave",
    "Pension plan",
]

# Run the agent
columns_to_drop = run_feature_selection_agent(columns_to_evaluate)

print("\n--- Summary ---")
print(f"Columns selected for dropping by Agent: {columns_to_drop}")

# Execute the deterministic drop
if columns_to_drop:
    new_df = new_df.drop(*columns_to_drop)
    print("🗑️ Dropped weak features from DataFrame.")
else:
    print("👍 Agent decided to keep all candidate features.")


🚀 Starting Agent Feature Selection Loop...

🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'benefit_count'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'description_len'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'compensation_type'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'listed_year'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'listed_month'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'listed_dayofweek'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'Paid maternity leave'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'Paid paternity leave'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'Pension plan'}
🛠️ Agent calling `drop_feature` with arguments: {'columns_to_drop': ['compensation_type', 'listed_year'], 'ju

In [ ]:
import os
import json
import operator
from typing import TypedDict, Annotated, List
import pyspark.sql.functions as F

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. CONNECT TO OPENAI
# ==========================================
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('YOUR_OPENAI_API_KEY')

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set. Set it before running this cell.")

# ==========================================
# 2. DEFINE AGENT STATE (REQUIRED BY RUBRIC)
# ==========================================
# Helper function to append to lists in the state without duplicating
def append_unique_list(existing: List[str], new: List[str]) -> List[str]:
    if existing is None: existing = []
    if new is None: new = []
    return list(set(existing + new))

# Explicit state structure defining what information the agent tracks
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add] # Tracks conversation history
    candidate_columns: List[str]                         # Columns left to evaluate
    dropped_columns: Annotated[List[str], append_unique_list] # Columns selected for removal

# ==========================================
# 3. DEFINE LANGCHAIN TOOLS (PYSPARK SKILLS)
# ==========================================
@tool
def evaluate_feature_signal(column_name: str) -> str:
    """
    Agent Skill: Calculates missingness, variance, and signal strength
    against target_log_salary.
    """
    try:
        total_rows = df.count()
        col_type = dict(df.dtypes).get(column_name, "string")
        is_numeric = col_type in ('int', 'bigint', 'double', 'float', 'long')

        if is_numeric:
            null_count = df.filter(F.col(column_name).isNull() | F.isnan(F.col(column_name))).count()
        else:
            null_count = df.filter(F.col(column_name).isNull()).count()

        null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
        distinct_count = df.select(column_name).distinct().count()

        result = {
            "column": column_name,
            "type": col_type,
            "null_percentage": round(null_pct, 2),
            "distinct_values_count": distinct_count,
            "signal_type": "None",
            "signal_data": {}
        }

        if distinct_count > 1 and null_pct < 100:
            if is_numeric:
                corr = df.stat.corr(column_name, "target_log_salary")
                result["signal_type"] = "pearson_correlation"
                result["signal_data"] = {"correlation_with_target": round(corr, 4) if corr else 0.0}
            else:
                group_stats = (
                    df.filter(F.col(column_name).isNotNull())
                      .groupBy(column_name)
                      .agg(
                          F.count("*").alias("count"),
                          F.expr("percentile_approx(target_log_salary, 0.5)").alias("median_log_salary")
                      )
                      .orderBy(F.desc("count"))
                      .limit(10)
                      .collect()
                )

                medians = [
                    {
                        "category": str(row[column_name]),
                        "count": row["count"],
                        "median_log_salary": round(row["median_log_salary"], 3) if row["median_log_salary"] else None
                    }
                    for row in group_stats
                ]
                result["signal_type"] = "group_medians"
                result["signal_data"] = {"top_categories": medians}

        return json.dumps(result)
    except Exception as e:
        return json.dumps({"error": str(e)})

@tool
def drop_feature(columns_to_drop: List[str], justification: str) -> str:
    """
    Agent Skill: Marks columns for removal if they lack statistical signal or have high missingness.
    Call this when you have decided a column is not useful for the model.
    """
    return f"Successfully marked {columns_to_drop} for removal. Reason: {justification}"

tools = [evaluate_feature_signal, drop_feature]

# ==========================================
# 4. CONSTRUCT LANGGRAPH NODES
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: AgentState):
    """Invokes the LLM to decide the next action based on the state."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tools_node(state: AgentState):
    """Executes tools and updates the dropped_columns state if applicable."""
    last_message = state["messages"][-1]

    tool_responses = []
    dropped_this_turn = []

    for tool_call in last_message.tool_calls:
        action = tool_call["name"]
        args = tool_call["args"]

        print(f"🛠️ Agent calling `{action}` with arguments: {args}")

        # Route to appropriate tool
        if action == "evaluate_feature_signal":
            result = evaluate_feature_signal.invoke(args)
        elif action == "drop_feature":
            result = drop_feature.invoke(args)
            # Update our LangGraph state explicitly!
            dropped_this_turn.extend(args.get("columns_to_drop", []))
        else:
            result = f"Unknown tool: {action}"

        # Append the tool's response to the conversation history
        tool_responses.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    # LangGraph merges these returned keys into the main AgentState
    return {"messages": tool_responses, "dropped_columns": dropped_this_turn}

def should_continue(state: AgentState):
    """Routing function to determine if the agent is done or needs to run a tool."""
    last_message = state["messages"][-1]
    # If the LLM made a tool call, we must execute the tools node
    if last_message.tool_calls:
        return "tools"
    # Otherwise, the LLM is done and just outputting final text
    return END

# ==========================================
# 5. BUILD AND COMPILE THE STATEGRAPH
# ==========================================
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tools_node)

# Add Edges
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, ["tools", END])
workflow.add_edge("tools", "agent")

# Compile the Graph
feature_selection_app = workflow.compile()

# ==========================================
# 6. EXECUTE THE WORKFLOW
# ==========================================

# 1. First, manually drop columns known to be entirely redundant or leaky
redundant_columns_to_drop = [
    "target_salary", "formatted_experience_level", "location_clean", "city",
    "state", "work_type", "employee_count_missing", "employee_count_filled",
    "follower_count_missing", "follower_count_filled", "title_len",
    "title_word_count", "description_word_count", "skills_desc_len",
    "skills_desc_word_count", "location_word_count", "benefit_type_count",
    "has_inferred_benefit", "posting_domain"
]

new_df = df.drop(*redundant_columns_to_drop)

# 2. Define columns for the agent to evaluate statistically
columns_to_evaluate = [
    "benefit_count", "description_len", "compensation_type", "listed_year",
    "listed_month", "listed_dayofweek", "Paid maternity leave",
    "Paid paternity leave", "Pension plan"
]

system_prompt = f"""You are a Lead Data Scientist performing feature selection on a PySpark DataFrame.
Your goal is to evaluate the following candidate columns: {columns_to_evaluate} for predicting 'target_log_salary'.

Instructions:
1. Use 'evaluate_feature_signal' to inspect columns.
2. For numeric features, check pearson correlation. If it is close to 0 (e.g., between -0.05 and 0.05), it is a weak signal.
3. For categorical features, check the group medians. If the median target_log_salary is practically identical across all major groups, it is a weak signal.
4. Use 'drop_feature' to explicitly remove columns with weak signals, extremely high null rates (> 80%), or no variance (1 distinct value).
5. Once you have evaluated all candidate columns, provide a brief summary of your decisions to the user. Do not call any more tools after summarizing."""

# Initialize the state
initial_state = {
    "messages": [HumanMessage(content=system_prompt)],
    "candidate_columns": columns_to_evaluate,
    "dropped_columns": []  # Starts empty
}

print("🚀 Starting LangGraph Agent Feature Selection Loop...\n")

# Run the compiled LangGraph application
final_state = feature_selection_app.invoke(initial_state, config={"recursion_limit": 30})

# Extract the final list of dropped columns from the LangGraph state
final_dropped_columns = final_state.get("dropped_columns", [])

print("\n✅ Agent finished evaluation.")
print(f"Agent's final thought process:\n{final_state['messages'][-1].content}")

print("\n--- Summary ---")
print(f"Columns explicitly selected for dropping by Agent state tracker: {final_dropped_columns}")

# Execute the final deterministic drop on the PySpark DataFrame
if final_dropped_columns:
    new_df = new_df.drop(*final_dropped_columns)
    print("🗑️ Dropped weak features from DataFrame.")
else:
    print("👍 Agent decided to keep all candidate features.")

🚀 Starting LangGraph Agent Feature Selection Loop...

🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'benefit_count'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'description_len'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'compensation_type'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'listed_year'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'listed_month'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'listed_dayofweek'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'Paid maternity leave'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'Paid paternity leave'}
🛠️ Agent calling `evaluate_feature_signal` with arguments: {'column_name': 'Pension plan'}
🛠️ Agent calling `drop_feature` with arguments: {'columns_to_drop': ['compensation_type', 'listed_y

## 7. Final Dataset Export
Finally, we export our fully cleaned, feature-engineered, and agent-pruned dataset to both CSV and Parquet formats for easy handoff to the model training pipeline.

In [ ]:
FINAL_AGENT_CSV_PATH = "/content/drive/MyDrive/BT4221 Group 13/Dataset/final_agent_handoff_csv"
FINAL_AGENT_PARQUET_PATH = "/content/drive/MyDrive/BT4221 Group 13/Dataset/final_agent_handoff_parquet"
handoff_df = new_df
handoff_df.write.mode("overwrite").option("header", True).csv(FINAL_AGENT_CSV_PATH)
handoff_df.write.mode("overwrite").parquet(FINAL_AGENT_PARQUET_PATH)

print("\nSaved agent-based handoff dataset to:")
print("CSV folder:", FINAL_AGENT_CSV_PATH)
print("Parquet:", FINAL_AGENT_PARQUET_PATH)


Saved agent-based handoff dataset to:
CSV folder: /content/drive/MyDrive/BT4221 Group 13/Dataset/final_agent_handoff_csv
Parquet: /content/drive/MyDrive/BT4221 Group 13/Dataset/final_agent_handoff_parquet
